In [1]:
# Import Required Libraries
import numpy as np # numpy is used for numerical operations and mathematical calculations
import pandas as pd # Pandas is used for data loading, manipulation and analysis

# Model Selection & Preprocessing Utilities
from sklearn.model_selection import (
    train_test_split, # train_test_split is used to divide the dataset into training and testing sets
    GridSearchCV # GridSearchCV is used for hyperparameter tuning with cross-validation
)
from sklearn.compose import ColumnTransformer # ColumnTransformer allows different preprocessing for numerical and categorical columns
from sklearn.preprocessing import (
    OneHotEncoder, # OneHotEncoder converts categorical variables into numerical format
    StandardScaler # StandardScaler standardizes numerical features for better model performance
)
from sklearn.pipeline import Pipeline # Pipeline helps combine preprocessing and modeling steps in a clean workflow
from sklearn.impute import SimpleImputer # Import imputer to fill missing values

# Import classification models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Import evaluation metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Ignore unnecessary warnings
import warnings
warnings.filterwarnings("ignore")

### Dataset load and preview

In [2]:
df = pd.read_csv("diabetes.csv") # Read the diabetes dataset from CSV file into a pandas DataFrame

df.head() # Display the first five rows of the dataset

,Age,Gender,Polyuria,Polydipsia,sudden weight loss,weakness,Polyphagia,Genital thrush,visual blurring,Itching,Irritability,delayed healing,partial paresis,muscle stiffness,Alopecia,Obesity,class
0,40.0,Male,No,Yes,No,Yes,No,No,No,Yes,No,Yes,No,Yes,Yes,Yes,Positive
1,58.0,Male,No,No,No,Yes,No,No,Yes,No,No,No,Yes,No,Yes,No,Positive
2,NaN,Male,Yes,No,No,Yes,Yes,No,No,Yes,No,Yes,No,Yes,Yes,No,Positive
3,45.0,Male,No,No,Yes,Yes,Yes,Yes,No,Yes,No,Yes,No,No,No,No,Positive
4,600.0,Male,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Positive


### Outlier Remove

In [3]:
df = df[(df['Age'] > 0) & (df['Age'] < 120)] # Remove invalid age values

### Separate features and target

In [4]:
# Separate the input features (X) and the target variable (y)
X = df.drop("class", axis=1) # X has all the input columns
y = df['class'].map({'Positive':1, 'Negative':0}) # Convert target to numeric

### Column Type Identification

In [5]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns # Identify numerical columns (integer & float based features)
cat_cols = X.select_dtypes(include=['object']).columns # Identify categorical columns (object/string-based features)

### Numerical Feature Pipeline

In [6]:
# Pipeline for numerical data
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), # Fill missing values with median
    ('scaler', StandardScaler())                   # Scale numeric features
])

# Pipeline for categorical data
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')), # Fill missing with most common value
    ('onehot', OneHotEncoder(handle_unknown='ignore'))    # Convert categories to numbers
])

# Combine numeric and categorical preprocessing
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

### Train-Test Split

In [7]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### Model Initialization

In [8]:
# Define Logistic Regression model
cls_lr = LogisticRegression(max_iter=1000, random_state=42)

# Define Random Forest model
cls_rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Define Gradient Boosting model
cls_gbc = GradientBoostingClassifier(n_estimators=100, random_state=42)

### Model Dictionary

In [9]:
# Store all models in a dictionary for iterative training and evaluation
model_to_train = {
    'Logistic Regression': cls_lr,
    'Random Forest': cls_rf,
    'Gradient Boosting': cls_gbc
}

### Model Training & Evaluation Loop

In [10]:
result = [] # List to store model results

# Loop through each model in our dictionary
for name, model in model_to_train.items():
  # Create pipeline with preprocessing and model
  pipe = Pipeline(
      [
          ('preprocessor', preprocessor),
          ('model', model)
      ]
  )

  # Train the model
  pipe.fit(X_train, y_train)

  # Predict on test data
  y_pred = pipe.predict(X_test)

  # Store evaluation metrics
  result.append({
    "Model": name,
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-score": f1_score(y_test, y_pred)
  })

# Convert results to a DataFrame and sort by best accuracy
result_df = pd.DataFrame(result).sort_values("Accuracy", ascending=False)

# Show the comparison of all models
print(result_df)

                 Model  Accuracy  Precision    Recall  F1-score
1        Random Forest  1.000000   1.000000  1.000000  1.000000
2    Gradient Boosting  0.990385   1.000000  0.984375  0.992126
0  Logistic Regression  0.961538   0.983871  0.953125  0.968254


### Best Model Selection & Final Pipeline

In [11]:
# Get the name of the best model based on accuracy
best_model_name = result_df.iloc[0]['Model']

# Retrieve the actual model object using the name
best_model_obj = model_to_train[best_model_name]

# Create final pipeline using best model
final_pipe = Pipeline(
    steps=[
    ('preprocessor', preprocessor),
    ('model', best_model_obj)
])

### Hyperparameter Tuning with GridSearchCV

In [12]:
# Define the grid of hyperparameters for Random Forest
param_grid = {
    'model__n_estimators': [100, 200, 300],          # Number of trees
    'model__max_depth': [None, 5, 10, 15],           # Maximum tree depth
    'model__min_samples_split': [2, 5, 10],          # Minimum samples to split
    'model__min_samples_leaf': [1, 2, 4],            # Minimum samples per leaf
    'model__max_features': ['auto', 'sqrt', 'log2']  # Features per split
}

# Apply Grid Search with cross-validation
grid = GridSearchCV(
    final_pipe,
    param_grid,
    cv=5,                # 5-fold cross validation
    scoring='accuracy',  # Optimize accuracy
    n_jobs=-1            # use all CPU cores for faster training
)

# Fit GridSearchCV on training data to find the best parameters
grid.fit(X_train, y_train)

# Print the best combination of hyperparameters
print("Best Params:", grid.best_params_)

Best Params: {'model__max_depth': None, 'model__max_features': 'log2', 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 100}


### Best Model Extraction & Evaluation

In [13]:
best_model = grid.best_estimator_ # Get the best model after GridSearchCV tuning
y_pred = best_model.predict(X_test) # Predict on test data using the optimized model

# Calculate final evaluation metrics
accuracy = accuracy_score(y_test, y_pred)     # Overall correct diabetes predictions
precision = precision_score(y_test, y_pred)   # How many predicted diabetics are actually diabetic
recall = recall_score(y_test, y_pred)         # How many real diabetic patients were correctly found
f1 = f1_score(y_test, y_pred)                 # Balance between precision and recall

# Print metrics
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9903846153846154
Precision: 1.0
Recall: 0.984375
F1-score: 0.9921259842519685

Confusion Matrix:
 [[40  0]
 [ 1 63]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99        40
           1       1.00      0.98      0.99        64

    accuracy                           0.99       104
   macro avg       0.99      0.99      0.99       104
weighted avg       0.99      0.99      0.99       104

